# Fine-tuning the NER model on MEDDOCAN (T2.3)

Thin notebook: it clones the repository and calls `modeling/`; no logic lives here (design §5, ADR-0007).
Works from the Colab web UI or from VS Code with the Google Colab extension (kernel: Colab, GPU T4).

- At most **3 runs** (`train` refuses a 4th). An interrupted run is **resumed** by running the training cell again.
- Checkpoints every epoch and run logs go to Google Drive: `MyDrive/clinical-deid/runs/`.
- The final model is saved as **safetensors** in `run-<n>/model/`. It is never committed to git (published to the Hub in T2.5).
- Clear the outputs before committing changes to this notebook.

Data: MEDDOCAN v1.0 (Marimon et al., 2019), CC BY 4.0, DOI [10.5281/zenodo.4279323](https://doi.org/10.5281/zenodo.4279323).

In [ ]:
# 1. GPU check: training on CPU would take hours.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Persistent storage for checkpoints and run logs (survives a disconnected session).
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
RUNS_ROOT = Path("/content/drive/MyDrive/clinical-deid/runs")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
sorted(p.name for p in RUNS_ROOT.iterdir())

In [ ]:
# 3. Clone the repository (the code that trains is what is on GitHub, not local changes).
import os
import sys

REPO_URL = "https://github.com/agustinroviraquezada/clinical-deid-es.git"
REPO_REF = "main"  # branch, tag or commit to train from
REPO_DIR = "/content/clinical-deid-es"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin {REPO_REF} && git -C {REPO_DIR} checkout --quiet FETCH_HEAD
!git -C {REPO_DIR} log --oneline -1

os.chdir(REPO_DIR)
sys.path[:0] = [REPO_DIR, f"{REPO_DIR}/src"]

In [ ]:
# 4. Install the locked `train` dependency group into the Colab kernel.
# torch and its CUDA packages are filtered out: Colab already ships a matching GPU build.
!pip install --quiet uv
EXPORT_ARGS = "--frozen --only-group train --no-emit-project --no-hashes"
TORCH_FAMILY = "^(torch|triton|nvidia-|cuda-)"
!uv export {EXPORT_ARGS} | grep -v -E "{TORCH_FAMILY}" > /tmp/requirements-train.txt
!pip install --quiet -r /tmp/requirements-train.txt
!python -c "import torch, transformers; print(torch.__version__, transformers.__version__)"

In [ ]:
# 5. Download MEDDOCAN (md5 verified) and convert it to BIO sentences.
!python -m modeling.data_prep.download_meddocan --dest data
!python -m modeling.data_prep.brat_to_bio --corpus data/meddocan --out data/meddocan/bio

In [ ]:
# 6. Train. Change only the hyperparameters of the next run here; keep them unchanged to resume.
from modeling.training.train_ner import RunConfig, train

config = RunConfig(
    learning_rate=5e-5,
    num_train_epochs=4,
    per_device_train_batch_size=16,
)
config

In [ ]:
log_path = train(config, data_dir=Path("data/meddocan/bio"), runs_root=RUNS_ROOT)
print(log_path.read_text(encoding="utf-8"))